### Terminal Commands

In [10]:
%matplotlib qt

## Plots

### 2D Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import glob

# --- Configuration ---
CSV_FOLDER = 'csv_exports'
COLORS = {
    'Control': 'black',
    'CTL': 'black',
    'DKO': 'blue',
    'PTEN': 'red'
}
DEFAULT_COLOR = 'gray'

def get_color_from_filename(filename):
    """Determines color based on the filename."""
    name_lower = filename.lower()
    for key, color in COLORS.items():
        if key.lower() in name_lower:
            return color
    return DEFAULT_COLOR

def plot_group_arrows_2d(csv_path):
    # 1. Load Data
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Error reading {csv_path}: {e}")
        return

    # Check columns
    if 'LevelAdj' not in df.columns or 'mu_deg' not in df.columns or 'r' not in df.columns:
        print(f"Skipping {os.path.basename(csv_path)}: Missing required columns.")
        return

    # 2. Extract Group Info
    filename = os.path.basename(csv_path)
    group_name = os.path.splitext(filename)[0]
    group_color = get_color_from_filename(group_name)
    
    print(f"\nPlotting Group: {group_name}")

    # 3. Process Data by Layer
    layers = sorted(df['LevelAdj'].dropna().unique())
    
    vectors = []
    for layer_idx in layers:
        layer_data = df[df['LevelAdj'] == layer_idx].dropna(subset=['r', 'mu_deg'])
        
        if len(layer_data) == 0:
            continue
            
        # --- MATCHING MATLAB LOGIC ---
        # To get the "Unweighted Mean Phase" and "Resultant Vector Length (R)",
        # we must treat every data point as a unit vector (length = 1).
        # This ignores the amplitude 'r' of the individual points, focusing only on phase consistency.
        
        r = np.ones(len(layer_data)) # Force length to 1
        theta = np.deg2rad(layer_data['mu_deg'].values)
        
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        mean_x = np.mean(x)
        mean_y = np.mean(y)
        
        # Convert mean vector back to Polar
        # mean_r is now the Resultant Vector Length (R), ranging 0 to 1
        mean_r = np.sqrt(mean_x**2 + mean_y**2)
        mean_theta = np.arctan2(mean_y, mean_x)
        
        # --- PRINT STATS TO CONSOLE ---
        deg_val = np.degrees(mean_theta) % 360
        print(f"  L{int(layer_idx)}: Phase={deg_val:5.1f}°, R={mean_r:.3f}, n={len(layer_data)}")
        
        vectors.append({
            'layer': int(layer_idx),
            'theta': mean_theta,
            'r': mean_r
        })

    if not vectors:
        print(f"No valid data found for {group_name}")
        return

    # 4. Setup 2D Polar Plot
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
    
    # Set the direction of 0 degrees (Standard is East/Right)
    ax.set_theta_zero_location('E') 
    ax.set_theta_direction(1) # Counter-Clockwise

    # 5. Draw Arrows
    
    # Force R max to 1.0 (Since R-value is strictly 0 to 1)
    ax.set_ylim(0, 1.0) 

    for v in vectors:
        deg_val = np.degrees(v['theta']) % 360
        
        # Plot the arrow
        ax.annotate(
            '', 
            xy=(v['theta'], v['r']), 
            xytext=(0, 0), 
            arrowprops=dict(
                facecolor=group_color, 
                edgecolor=group_color,
                width=3,       
                headwidth=10,  
                alpha=0.8
            )
        )
        
        # Label with Layer and Degree
        label_text = f"L{v['layer']}\n{deg_val:.0f}°"
        
        # Position label slightly beyond the arrow tip
        text_r = v['r'] + 0.1
        
        ax.text(
            v['theta'], text_r, 
            label_text, 
            color=group_color, 
            fontsize=9, 
            fontweight='bold',
            ha='center', va='center'
        )

    # 6. Styling
    ax.set_title(f"Mean Phase & Consistency (R): {group_name}", fontsize=15, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

# --- Main Execution ---
if __name__ == "__main__":
    csv_files = glob.glob(os.path.join(CSV_FOLDER, "*.csv"))
    
    if not csv_files:
        print(f"No CSV files found in '{CSV_FOLDER}'")
    else:
        for csv_file in csv_files:
            plot_group_arrows_2d(csv_file)

### 3D Plots

In [17]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import glob
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d import proj3d

# --- Configuration ---
CSV_FOLDER = 'csv_exports'

# 1. Dimensions & Zoom
LAYER_SPACING = 3.0      # Physical distance between layers
Z_AXIS_STRETCH = 2.5     # Visual stretch
R_LIMIT = 1.0            # Set to 1.0 to fit R-value arrows (0-1 range)

# 2. Styling
DOT_TRANSPARENCY = 0.3
ARROW_ALPHA = 0.9
ARROW_SHAFT_WIDTH = 3    # Standard width
ARROW_HEAD_WIDTH = 20    # Standard head scale
R_THRESHOLD = 0.3333     # <--- New Threshold for scaling

# 3. View Settings
DEFAULT_ELEV = 15
DEFAULT_AZIM = -95

# 4. Genotype Colors
GENOTYPE_COLORS = {
    'Control': 'black',
    'CTL': 'black',
    'DKO': 'blue',
    'PTEN': 'red'
}
DEFAULT_COLOR = 'gray'

# --- Helper Function: Rayleigh Test ---
def rayleigh_p_approx(R, n):
    if n <= 0 or not np.isfinite(R):
        return np.nan
    Z = n * (R**2)
    p = np.exp(-Z) * (1 + (2*Z - Z**2) / (4*n) - 
                      (24*Z - 132*Z**2 + 76*Z**3 - 9*Z**4) / (288*n**2))
    return max(0, min(p, 1))

# --- Custom Class for 3D Arrows ---
class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0,0), (0,0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
        self.set_positions((xs[0],ys[0]),(xs[1],ys[1]))
        return np.min(zs)

def get_color_from_filename(filename):
    name_lower = filename.lower()
    for key, color in GENOTYPE_COLORS.items():
        if key.lower() in name_lower:
            return color
    return DEFAULT_COLOR

def plot_layer_data_final(csv_path):
    # 1. Load Data
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Error reading {csv_path}: {e}")
        return

    filename = os.path.basename(csv_path)
    group_name = os.path.splitext(filename)[0]
    group_color = get_color_from_filename(group_name)
    
    print(f"\nPlotting 3D: {group_name}")

    # 2. Setup 3D Plot
    fig = plt.figure(figsize=(12, 12)) 
    ax = fig.add_subplot(111, projection='3d')
    ax.view_init(elev=DEFAULT_ELEV, azim=DEFAULT_AZIM)

    if 'LevelAdj' not in df.columns:
        print(f"Skipping {filename}: 'LevelAdj' missing.")
        return

    layers = sorted(df['LevelAdj'].dropna().unique())
    if not layers: return
    
    # Calculate Z mapping to flip order (L1 on Top)
    max_layer_idx = max(layers)
    top_z = (max_layer_idx - min(layers) + 1) * LAYER_SPACING

    # --- MAIN LOOP: PLOT LAYERS ---
    for layer_idx in layers:
        layer_data = df[df['LevelAdj'] == layer_idx].dropna(subset=['r', 'mu_deg'])
        if len(layer_data) == 0:
            continue
        
        # Count N
        n_val = len(layer_data)

        # Calculate Statistics (Unit Vector Method)
        r_unit = np.ones(len(layer_data))
        theta_rad = np.deg2rad(layer_data['mu_deg'].values)
        
        x_unit = r_unit * np.cos(theta_rad)
        y_unit = r_unit * np.sin(theta_rad)
        
        mean_x = np.mean(x_unit)
        mean_y = np.mean(y_unit)
        
        R_val = np.sqrt(mean_x**2 + mean_y**2)
        mu_deg = np.degrees(np.arctan2(mean_y, mean_x)) % 360
        p_val = rayleigh_p_approx(R_val, n_val)
        
        print(f"  L{int(layer_idx)}: Phase={mu_deg:5.1f}°, R={R_val:.3f}, n={n_val}")
        
        # Z Height (Flipped)
        z_height = (max_layer_idx - layer_idx + 1) * LAYER_SPACING
        
        # A. Plot DOTS
        r_real = layer_data['r'].values
        x_real = r_real * np.cos(theta_rad)
        y_real = r_real * np.sin(theta_rad)
        
        mask = r_real <= R_LIMIT
        x_plot = x_real[mask]
        y_plot = y_real[mask]
        z_plot = np.full_like(x_plot, z_height)
        
        ax.scatter(x_plot, y_plot, z_plot, 
                   c=group_color, 
                   alpha=DOT_TRANSPARENCY, 
                   s=25, 
                   edgecolors='none')
        
        # B. Plot ARROW (DYNAMIC SCALING)
        if n_val > 0:
            # --- New Logic: Dynamic Scaling based on R_val ---
            if R_val < R_THRESHOLD:
                # Scale down linearly from threshold
                scale_factor = R_val / R_THRESHOLD 
                # Ensure it doesn't disappear completely, set a minimum scale
                scale_factor = max(scale_factor, 0.2) 
                
                dynamic_shaft = ARROW_SHAFT_WIDTH * scale_factor
                dynamic_head = ARROW_HEAD_WIDTH * scale_factor
            else:
                # Use standard sizes
                dynamic_shaft = ARROW_SHAFT_WIDTH
                dynamic_head = ARROW_HEAD_WIDTH

            # 1. Background White Arrow (Outline)
            # Note: outline is slightly thicker than the shaft
            a_bg = Arrow3D([0, mean_x], [0, mean_y], [z_height, z_height], 
                        mutation_scale=dynamic_head, 
                        lw=dynamic_shaft + 2.0 * (scale_factor if R_val < R_THRESHOLD else 1), 
                        arrowstyle="-|>", 
                        color='white', 
                        alpha=1.0)
            ax.add_artist(a_bg)
            
            # 2. Foreground Colored Arrow
            a_fg = Arrow3D([0, mean_x], [0, mean_y], [z_height, z_height], 
                        mutation_scale=dynamic_head, 
                        lw=dynamic_shaft, 
                        arrowstyle="-|>", 
                        color=group_color, 
                        alpha=ARROW_ALPHA)
            ax.add_artist(a_fg)

        # C. Stats Text
        if p_val < 0.05: p_str = "p<0.05"
        else: p_str = f"p={p_val:.2f}"
            
        stats_text = (f"\n"  
                      f"n={n_val}\n"
                      f"μ={mu_deg:.0f}°\n"
                      f"R={R_val:.2f}\n"
                      f"{p_str}")
        
        text_offset = R_LIMIT * 1.2
        nx = text_offset * np.cos(np.deg2rad(315)) 
        ny = text_offset * np.sin(np.deg2rad(315))
        ax.text(nx, ny, z_height, stats_text, 
                color='black', fontsize=9, ha='left', va='center', linespacing=1.2)

        # D. Grid Rings & Lines
        t_grid = np.linspace(0, 2*np.pi, 100)
        ring_radii = [R_LIMIT/2, R_LIMIT] 
        for r_val in ring_radii:
            xg = r_val * np.cos(t_grid)
            yg = r_val * np.sin(t_grid)
            zg = np.full_like(xg, z_height)
            ax.plot(xg, yg, zg, color='gray', alpha=0.2, linewidth=0.8)
            ax.text(r_val, 0.02, z_height, f"{r_val:.1f}", fontsize=8, color='black', ha='center')

        deg_labels = {0: '0°', 90: '90°', 180: '180°', 270: '270°'}
        for deg, text in deg_labels.items():
            rad = np.deg2rad(deg)
            ax.plot([0, R_LIMIT * np.cos(rad)], [0, R_LIMIT * np.sin(rad)], [z_height, z_height], 
                    color='gray', alpha=0.15, linewidth=0.8)
            xt = (R_LIMIT * 1.15) * np.cos(rad)
            yt = (R_LIMIT * 1.15) * np.sin(rad)
            ax.text(xt, yt, z_height, text, fontsize=8, color='black', ha='center', va='center')

    # --- 3D Grid Cage ---
    ax.plot([0, 0], [0, 0], [0, top_z + LAYER_SPACING], 
            color='black', alpha=0.1, linewidth=1, linestyle='-')

    cage_angles = [0, 90, 180, 270]
    for deg in cage_angles:
        rad = np.deg2rad(deg)
        cx = R_LIMIT * np.cos(rad)
        cy = R_LIMIT * np.sin(rad)
        ax.plot([cx, cx], [cy, cy], [0, top_z + LAYER_SPACING], 
                color='gray', alpha=0.1, linewidth=0.8, linestyle='--')

    # 3. Styling
    ax.set_title(f'{group_name}', fontsize=18, fontweight='bold', pad=10, color=group_color)
    ax.set_axis_off() 
    ax.set_xlim(-R_LIMIT, R_LIMIT)
    ax.set_ylim(-R_LIMIT, R_LIMIT)
    ax.set_zlim(0, top_z + LAYER_SPACING)
    ax.set_box_aspect((1, 1, Z_AXIS_STRETCH))
    
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # Create dummy data for the script to run
    os.makedirs(CSV_FOLDER, exist_ok=True)
    
    # Create a dummy CSV file if one doesn't exist
    dummy_csv_path = os.path.join(CSV_FOLDER, 'dummy_data.csv')
    if not os.path.exists(dummy_csv_path):
        # Create a sample DataFrame
        # Layer 3 is intentionally made very random to have a small R
        data = {
            'LevelAdj': [1]*50 + [2]*50 + [3]*50 + [4]*50,
            'r': np.random.uniform(0.1, 0.9, 200),
            'mu_deg': np.concatenate([
                np.random.normal(45, 20, 50),  # L1: Consistent-ish
                np.random.normal(135, 60, 50), # L2: Less consistent
                np.random.uniform(0, 360, 50), # L3: Random -> Small R
                np.random.normal(270, 15, 50)  # L4: Very consistent
            ])
        }
        df_dummy = pd.DataFrame(data)
        df_dummy.to_csv(dummy_csv_path, index=False)
        print(f"Created dummy data file: {dummy_csv_path}")

    # Run the plotting function
    csv_files = glob.glob(os.path.join(CSV_FOLDER, "*.csv"))
    if not csv_files:
        print(f"No CSV files found in '{CSV_FOLDER}'")
    else:
        for csv_file in csv_files:
            plot_layer_data_final(csv_file)


Plotting 3D: CTL_CA1
  L1: Phase= 14.0°, R=0.301, n=82
  L2: Phase= 36.7°, R=0.268, n=82
  L3: Phase= 62.4°, R=0.273, n=82
  L4: Phase= 75.7°, R=0.266, n=81

Plotting 3D: CTL_DG
  L1: Phase= 97.5°, R=0.077, n=67
  L2: Phase=139.0°, R=0.052, n=68
  L3: Phase= 81.8°, R=0.032, n=68
  L4: Phase=215.4°, R=0.078, n=67

Plotting 3D: DKO_CA1
  L1: Phase=256.8°, R=0.289, n=66
  L2: Phase=315.6°, R=0.303, n=66
  L3: Phase=354.7°, R=0.459, n=66
  L4: Phase= 67.0°, R=0.331, n=66

Plotting 3D: DKO_DG
  L1: Phase=338.4°, R=0.174, n=52
  L2: Phase=321.8°, R=0.238, n=52
  L3: Phase=318.8°, R=0.439, n=52
  L4: Phase=308.7°, R=0.431, n=51

Plotting 3D: PTEN_CA1
  L1: Phase=290.3°, R=0.160, n=39
  L2: Phase=252.3°, R=0.186, n=39
  L3: Phase=245.4°, R=0.187, n=39
  L4: Phase=245.7°, R=0.178, n=39

Plotting 3D: PTEN_DG
  L1: Phase= 19.2°, R=0.573, n=55
  L2: Phase=  2.3°, R=0.588, n=57
  L3: Phase=329.5°, R=0.526, n=57
  L4: Phase=317.2°, R=0.449, n=57
